# 01 — Data and preprocessing

Before any model can learn, the data has to be *clean*, *consistent*, and *the right size*. This notebook walks through every preprocessing step the project performs, the math behind each, and how to verify it with code.

## Roadmap

1. The datasets we use, and why
2. The shape of one training example
3. **Why downscale?** — memory math
4. **How we downscale** — LANCZOS resampling explained
5. **Quality scoring** — variance-of-Laplacian for blur, brightness, contrast
6. Augmentation: RandAugment, MixUp, CutMix — visualised


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## 1. Datasets

| Dataset | Used for | Source |
|---|---|---|
| **CarDD** (Wang et al. 2023) | Damage classifier + detector training | Kaggle: `eduardo4jesus/cardd` |
| **Stanford Cars** | Car make/model identifier | Kaggle: `eduardo4jesus/stanford-cars-dataset` |

The cost target is **synthetic** — derived from a versioned parts-cost catalog × car age × a small Gaussian noise term. There is no public dataset that pairs damaged-car photos with real repair invoices, and inventing one would be dishonest. We document this trade-off in `PLAN.md §3`.

## 2. One training example

```
{
  "image":     <PIL.Image  H×W×3 RGB>,
  "damage":    ["dent", "scratch"],          # multi-label
  "bboxes":    [BBox(damage_type="dent", x_center=..., y_center=..., width=..., height=...)],
  "make":      "toyota",
  "model":     "camry",
  "year":      2015,
  "cost_usd":  812.50,                       # synthetic target
}
```


In [ ]:
# Peek at the schema in code form.
from ccdp.data.schema import DAMAGE_TYPES, BBox, infer_part_from_damage
import inspect

print("Damage classes:", DAMAGE_TYPES)
print()
print("BBox fields:")
print(inspect.getsource(BBox))


## 3. Why downscale?

A modern phone shoots photos at ~12 megapixels. Stored as RGB uint8 that is:

$$
12{,}000{,}000 \text{ pixels} \times 3 \text{ channels} \times 1 \text{ byte} = 36 \text{ MB per image}
$$

For a batch of 32, that's **1.15 GB** of pixels — more than free-Colab T4 VRAM headroom after the model is loaded. And **the model never sees that resolution anyway**: ResNet50 inputs are 224×224, YOLOv8 is 640×640. So we resize once, up front, to a long edge of **1600 px**.


In [ ]:
# Concrete memory comparison.
def bytes_for(w, h, c=3, dtype_bytes=1):
    return w * h * c * dtype_bytes

raw_phone   = bytes_for(4000, 3000)
downscaled  = bytes_for(1600, 1200)
classifier  = bytes_for(224, 224)
detector    = bytes_for(640, 640)

print(f"Raw phone photo (4000x3000): {raw_phone / 1e6:6.1f} MB")
print(f"Downscaled  (1600x1200):     {downscaled / 1e6:6.1f} MB")
print(f"Classifier input (224x224):  {classifier / 1e3:6.1f} KB")
print(f"Detector input (640x640):    {detector / 1e3:6.1f} KB")


## 4. How we downscale — LANCZOS resampling

When you shrink an image, you have to pick which pixels of the original to keep. The naive way is **nearest-neighbour**: for each output pixel, copy the closest input pixel. It's fast but creates jagged edges — bad for our models, which are trying to detect dent and scratch *edges*.

**LANCZOS** is a higher-order filter that blends multiple input pixels using the function

$$
L(x) = \begin{cases} \text{sinc}(x)\,\text{sinc}(x/a) & |x| < a \\ 0 & \text{otherwise}\end{cases}
$$

where $\text{sinc}(x) = \sin(\pi x) / (\pi x)$ and $a$ (the *kernel size*) is typically 3. The result preserves edges better than nearest-neighbour or bilinear, which is why Pillow recommends it for downsizing.

Let's see the difference visually.


In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Build a synthetic high-frequency test pattern (alternating black/white bars).
big = np.zeros((400, 400, 3), dtype=np.uint8)
big[:, ::4] = 255  # vertical stripes every 4 pixels
img = Image.fromarray(big)

nearest = img.resize((100, 100), resample=Image.NEAREST)
bilinear = img.resize((100, 100), resample=Image.BILINEAR)
lanczos = img.resize((100, 100), resample=Image.LANCZOS)

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for ax, im, title in zip(axes, [img, nearest, bilinear, lanczos],
                          ["original 400x400", "NEAREST", "BILINEAR", "LANCZOS"]):
    ax.imshow(im); ax.set_title(title); ax.axis("off")
plt.show()


LANCZOS produces the smoothest stripes — the others alias into Moiré patterns. Now let's call the project's own downscaler.


In [ ]:
from ccdp.preprocess.pipeline import normalize_for_inference, quality_report
from PIL import Image
import numpy as np

# Make a fake "phone photo" — 4000x3000 random noise just for size demo.
fake = Image.fromarray(np.random.randint(0, 255, (3000, 4000, 3), dtype=np.uint8))
print("input :", fake.size)
small = normalize_for_inference(fake, max_long_edge=1600)
print("output:", small.size)
print("note: long edge clamped to 1600, aspect ratio preserved.")


## 5. Quality scoring — is the image even usable?

If a user uploads a phone photo so blurry that even a human can't tell a dent from a reflection, the model will guess wildly and the cost estimate will be junk. So we **measure** image quality and surface it in the API response.

### Variance of the Laplacian (the standard blur metric)

The Laplacian is a 2D second-derivative operator. Apply this 3×3 kernel as a convolution to a greyscale image:

$$
\nabla^2 = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}
$$

For a sharp image with crisp edges, the result has *high variance* — lots of strongly-positive and strongly-negative responses near edges. A blurry image has *low variance* because edges are smeared.

This single number — `Var(∇² image)` — is the textbook "is it blurry?" metric (Pech-Pacheco et al., 2000).

Let's reproduce it.


In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ccdp.preprocess.pipeline import _sharpness_score, quality_report

# Build a sharp image (edges) and a blurry copy of the same.
sharp_arr = np.zeros((256, 256), dtype=np.uint8)
sharp_arr[64:192, 64:192] = 255    # crisp white square on black
sharp = Image.fromarray(sharp_arr).convert("RGB")

blurry = sharp.filter(__import__("PIL.ImageFilter", fromlist=["GaussianBlur"]).GaussianBlur(radius=8))

print(f"Sharpness  sharp: {_sharpness_score(sharp):8.1f}")
print(f"Sharpness blurry: {_sharpness_score(blurry):8.1f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sharp); axes[0].set_title("sharp"); axes[0].axis("off")
axes[1].imshow(blurry); axes[1].set_title("blurry"); axes[1].axis("off")
plt.show()


In [ ]:
# The full quality report the API returns:
quality_report(sharp)


## 6. Augmentation — making models robust

A model that has only ever seen well-lit, centered, crisp photos will fail on real user uploads. We *augment* the training set by randomly distorting each image differently each epoch. Three popular techniques:

- **RandAugment** — pick N random transforms (rotate, color jitter, posterize…) from a list and chain them. Reduces hyperparameter tuning.
- **MixUp** — for two images $x_i, x_j$ with labels $y_i, y_j$, train on $(\lambda x_i + (1-\lambda) x_j,\ \lambda y_i + (1-\lambda) y_j)$ where $\lambda \sim \text{Beta}(\alpha, \alpha)$.
- **CutMix** — paste a random rectangle from image B onto image A; mix labels by the area ratio.

The intuition: these force the model to make decisions based on *content* rather than memorising exact pixel arrangements.


In [ ]:
# Visualise MixUp on two synthetic images.
import numpy as np
import matplotlib.pyplot as plt

img_a = np.zeros((128, 128, 3), dtype=np.float32); img_a[..., 0] = 1.0   # red
img_b = np.zeros((128, 128, 3), dtype=np.float32); img_b[..., 2] = 1.0   # blue

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, lam in zip(axes, [1.0, 0.75, 0.5, 0.25, 0.0]):
    mix = lam * img_a + (1 - lam) * img_b
    ax.imshow(mix); ax.set_title(f"λ={lam}"); ax.axis("off")
plt.suptitle("MixUp: linear blend between two images and their labels")
plt.show()


**Next:** open `02_classifier_resnet50.ipynb` to learn what a convolutional network actually computes, why ResNet50 has *skip connections*, and how to train one yourself.
